In [1]:
!pip install requests beautifulsoup4 pandas openpyxl --quiet

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import os

headers = {"User-Agent": "Mozilla/5.0"}

def qiymetleri_tap(tag):
    current = tag
    for _ in range(6):
        current = current.find_parent()
        if current is None:
            break
        text = current.get_text(" ", strip=True)
        qiymetler = re.findall(r"(\d+\.\d{2})\s*AZN", text)
        if len(qiymetler) >= 1:
            return qiymetler
    return []

kateqoriyalar = {
    "Bestseller": "bestseller-3",
    "Detektiv. Triller": "detektivy-trillery-2",
    "Fantastika": "fantastika-uzhasy",
    "Sevgi Romanı": "lyubovnye-romany-2",
    "Dünya Klassikası": "mirovaya-klassika",
    "Poeziya": "poeziya-2",
}

butun_kitablar = []

for janr_adi, slug in kateqoriyalar.items():
    page = 1
    while True:
        url = f"https://alinino.az/collection/{slug}?page={page}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a", href=lambda x: x and "/product/" in x and "#" not in x)
        if not links:
            break

        seen_this_page = set()
        yeni_elave = 0
        for link_tag in links:
            title = link_tag.get_text(strip=True)
            href = link_tag["href"]
            if not title or title == "Sürətli görünüş" or title.isdigit():
                continue
            if href in seen_this_page:
                continue
            seen_this_page.add(href)

            qiymetler = qiymetleri_tap(link_tag)
            qiymet_yeni = qiymetler[0] if len(qiymetler) > 0 else None
            qiymet_kohne = qiymetler[1] if len(qiymetler) > 1 else None

            butun_kitablar.append({
                "Janr": janr_adi,
                "Başlıq": title,
                "Yeni qiymət (AZN)": qiymet_yeni,
                "Köhnə qiymət (AZN)": qiymet_kohne,
                "Link": "https://alinino.az" + href if href.startswith("/") else href
            })
            yeni_elave += 1

        print(f"{janr_adi} - səhifə {page}: {yeni_elave} kitab")

        if page >= 7:
            break
        page += 1
        time.sleep(1)

df = pd.DataFrame(butun_kitablar)
df = df.drop_duplicates(subset="Link")

# Excel faylı yaradıb birbaşa Desktop-a yazırıq
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
hedef_yol = os.path.join(desktop, "janrli_kitablar.xlsx")
df.to_excel(hedef_yol, index=False)

print(f"\nCƏMİ {len(df)} kitab tapıldı")
print(f"Excel faylı Desktop-a yazıldı: {hedef_yol}")
df.head(10)

Bestseller - səhifə 1: 47 kitab
Bestseller - səhifə 2: 47 kitab
Bestseller - səhifə 3: 47 kitab
Bestseller - səhifə 4: 47 kitab
Bestseller - səhifə 5: 47 kitab
Bestseller - səhifə 6: 47 kitab
Bestseller - səhifə 7: 32 kitab
Detektiv. Triller - səhifə 1: 56 kitab
Detektiv. Triller - səhifə 2: 56 kitab
Detektiv. Triller - səhifə 3: 56 kitab
Detektiv. Triller - səhifə 4: 56 kitab
Detektiv. Triller - səhifə 5: 56 kitab
Detektiv. Triller - səhifə 6: 56 kitab
Detektiv. Triller - səhifə 7: 56 kitab
Fantastika - səhifə 1: 56 kitab
Fantastika - səhifə 2: 56 kitab
Fantastika - səhifə 3: 56 kitab
Fantastika - səhifə 4: 56 kitab
Fantastika - səhifə 5: 56 kitab
Fantastika - səhifə 6: 56 kitab
Fantastika - səhifə 7: 56 kitab
Sevgi Romanı - səhifə 1: 49 kitab
Sevgi Romanı - səhifə 2: 49 kitab
Sevgi Romanı - səhifə 3: 49 kitab
Sevgi Romanı - səhifə 4: 29 kitab
Sevgi Romanı - səhifə 5: 9 kitab
Sevgi Romanı - səhifə 6: 9 kitab
Sevgi Romanı - səhifə 7: 9 kitab
Dünya Klassikası - səhifə 1: 52 kitab
Dünya 

,Janr,Başlıq,Yeni qiymət (AZN),Köhnə qiymət (AZN),Link
0,Bestseller,Min möhtəşəm günəş,9.59,11.99,https://alinino.az/product/min-mohtesem-gunes-3
1,Bestseller,Sirli bağ,8.49,9.99,https://alinino.az/product/sirli-bag-fd9d43
2,Bestseller,Odisseya,11.89,13.99,https://alinino.az/product/odisseya-5d4097
3,Bestseller,Leyləklərin uçuşu,15.26,17.95,https://alinino.az/product/leyleklerin-ucusu
4,Bestseller,Məktubdakı qadın,11.01,12.95,https://alinino.az/product/mektubdaki-qadin-92...
5,Bestseller,Mirvari sırğalı qız,11.86,13.95,https://alinino.az/product/mirvari-sirgali-qiz
6,Bestseller,Zəfər şəhəri,16.11,18.95,https://alinino.az/product/zefer-seheri
7,Bestseller,Yevgeni Onegin,8.49,9.99,https://alinino.az/product/yevgeni-onegin-2
8,Bestseller,Gecə musiqisi,12.71,14.95,https://alinino.az/product/gece-musiqisi
9,Bestseller,Atuan türbələri,9.31,10.95,https://alinino.az/product/atuan-turbeleri


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import os

headers = {"User-Agent": "Mozilla/5.0"}

def qiymetleri_tap(tag):
    current = tag
    for _ in range(6):
        current = current.find_parent()
        if current is None:
            break
        text = current.get_text(" ", strip=True)
        qiymetler = re.findall(r"(\d+\.\d{2})\s*AZN", text)
        if len(qiymetler) >= 1:
            return qiymetler
    return []

def sekil_tap(tag):
    current = tag
    for _ in range(6):
        current = current.find_parent()
        if current is None:
            break
        img = current.find("img")
        if img:
            src = img.get("src") or img.get("data-src")
            if src:
                if src.startswith("//"):
                    src = "https:" + src
                return src
    return None

kateqoriyalar = {
    "Bestseller": "bestseller-3",
    "Tarixi Roman": "istoricheskie-romany",
    "Sevgi Romanı": "lyubovnye-romany",
    "Poeziya": "poeziya",
    "Fantastika. Mistika": "fantastika-mistika",
    "Müasir Azərbaycan Ədəbiyyatı": "sovremennaya-azerbaydzhanskaya-literatura",
    "Dünya Klassikası": "mirovaya-i-azerbaydzhanskaya-klassika",
}

butun_kitablar = []

for janr_adi, slug in kateqoriyalar.items():
    page = 1
    while True:
        url = f"https://alinino.az/collection/{slug}?page={page}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a", href=lambda x: x and "/product/" in x and "#" not in x)
        if not links:
            break

        seen_this_page = set()
        yeni_elave = 0
        for link_tag in links:
            title = link_tag.get_text(strip=True)
            href = link_tag["href"]
            if not title or title == "Sürətli görünüş" or title.isdigit():
                continue
            if href in seen_this_page:
                continue
            seen_this_page.add(href)

            qiymetler = qiymetleri_tap(link_tag)
            qiymet_yeni = qiymetler[0] if len(qiymetler) > 0 else None
            qiymet_kohne = qiymetler[1] if len(qiymetler) > 1 else None
            sekil = sekil_tap(link_tag)

            butun_kitablar.append({
                "Janr": janr_adi,
                "Başlıq": title,
                "Yeni qiymət (AZN)": qiymet_yeni,
                "Köhnə qiymət (AZN)": qiymet_kohne,
                "Şəkil": sekil,
                "Link": "https://alinino.az" + href if href.startswith("/") else href
            })
            yeni_elave += 1

        print(f"{janr_adi} - səhifə {page}: {yeni_elave} kitab")

        if page >= 7:
            break
        page += 1
        time.sleep(1)

df = pd.DataFrame(butun_kitablar)
df = df.drop_duplicates(subset="Link")
df = df.dropna(subset=["Yeni qiymət (AZN)"])

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
hedef_yol = os.path.join(desktop, "az_kitablar_sekilli.xlsx")
df.to_excel(hedef_yol, index=False)

print(f"\nCƏMİ {len(df)} kitab tapıldı")
print(f"Excel faylı Desktop-a yazıldı: {hedef_yol}")
df.head(5)

Bestseller - səhifə 1: 47 kitab
Bestseller - səhifə 2: 47 kitab
Bestseller - səhifə 3: 47 kitab
Bestseller - səhifə 4: 47 kitab
Bestseller - səhifə 5: 47 kitab
Bestseller - səhifə 6: 47 kitab
Bestseller - səhifə 7: 32 kitab
Tarixi Roman - səhifə 1: 46 kitab
Tarixi Roman - səhifə 2: 46 kitab
Tarixi Roman - səhifə 3: 13 kitab
Tarixi Roman - səhifə 4: 6 kitab
Tarixi Roman - səhifə 5: 6 kitab
Tarixi Roman - səhifə 6: 6 kitab
Tarixi Roman - səhifə 7: 6 kitab
Sevgi Romanı - səhifə 1: 46 kitab
Sevgi Romanı - səhifə 2: 46 kitab
Sevgi Romanı - səhifə 3: 46 kitab
Sevgi Romanı - səhifə 4: 22 kitab
Sevgi Romanı - səhifə 5: 6 kitab
Sevgi Romanı - səhifə 6: 6 kitab
Sevgi Romanı - səhifə 7: 6 kitab
Poeziya - səhifə 1: 40 kitab
Poeziya - səhifə 2: 40 kitab
Poeziya - səhifə 3: 40 kitab
Poeziya - səhifə 4: 3 kitab
Fantastika. Mistika - səhifə 1: 45 kitab
Fantastika. Mistika - səhifə 2: 39 kitab
Fantastika. Mistika - səhifə 3: 5 kitab
Fantastika. Mistika - səhifə 4: 5 kitab
Fantastika. Mistika - səhifə 5

,Janr,Başlıq,Yeni qiymət (AZN),Köhnə qiymət (AZN),Şəkil,Link
0,Bestseller,Min möhtəşəm günəş,9.59,11.99,https://cdn.insales-shop.ru/r/zqj6VEFu4F4/rs:f...,https://alinino.az/product/min-mohtesem-gunes-3
1,Bestseller,Sirli bağ,8.49,9.99,https://cdn.insales-shop.ru/r/DRH9fkRoNTE/rs:f...,https://alinino.az/product/sirli-bag-fd9d43
2,Bestseller,Odisseya,11.89,13.99,https://cdn.insales-shop.ru/r/BDF6JwrwYe4/rs:f...,https://alinino.az/product/odisseya-5d4097
3,Bestseller,Leyləklərin uçuşu,15.26,17.95,https://cdn.insales-shop.ru/r/1TMbRs_Tz1c/rs:f...,https://alinino.az/product/leyleklerin-ucusu
4,Bestseller,Məktubdakı qadın,11.01,12.95,https://cdn.insales-shop.ru/r/K_ZvVfyah6s/rs:f...,https://alinino.az/product/mektubdaki-qadin-92...
